# our method synthetic dataset training

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Fit Zoetrope and integrate the Fisher flow

In [ ]:
# Selected Flow-STP Stage 1 and derived Stage 2 across ten observed tissues.
# Stage 1 uses the repository defaults; Stage 2 uses the fitted Fisher field.

from __future__ import annotations

import contextlib
import hashlib
import importlib.metadata
import io
import sys
import json
import os
import subprocess
import time
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import flow_stp
from flow_stp.stage1 import METHOD_VERSION, Stage1ARunConfig, run_stage1a
from flow_stp.stage2 import estimate_local_dynamics, integrate_flow_rk4

PINNED_COMMIT = "76bb952a841c3671ec35161149e780167fd9b7b1"
PROJECT = Path(os.environ.get(
    "STP_PROJECT", os.environ["STP_PROJECT"]
)).expanduser()
DATA = Path(os.environ.get(
    "CRESCENT_DATA_ROOT", str(PROJECT / "Experiments/simulation validation/"
                                      "Synthetic benchmarking/experiments/"
                                      "synthetic_developmental_crescent_v1")
)).expanduser()
OUT = Path(os.environ.get(
    "FLOW_STP_ALL_TEN_OUT", str(PROJECT / "experiments/synthetic_benchmark/"
                                  "synthetic_developmental_crescent_v1_flow_stp/"
                                  "all_ten_tissues_stage1_stage2_v1")
)).expanduser()
NATIVE = OUT / "stage1_native"
GENES = [f"G{i:02d}" for i in range(1, 49)]
STEPS = 128                      # Native Stage-2 RK4 default
EPS = 1e-8                      # Native Fisher projection denominator
CELL_BATCH = 512                # Only changes execution memory, not equations


def _sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def _json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".partial")
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + "\n")
    os.replace(tmp, path)


def _git(repo, *args):
    return subprocess.run(["git", "-C", str(repo), *args], check=True,
                          capture_output=True, text=True).stdout.strip()


def _check_source():
    package_dir = Path(flow_stp.__file__).resolve().parent
    assert package_dir.name == "flow_stp"
    repo = package_dir.parents[1]
    assert (repo / ".git").exists(), f"Not an editable checkout: {repo}"
    assert _git(repo, "rev-parse", "HEAD") == PINNED_COMMIT
    assert not _git(repo, "status", "--porcelain", "--untracked-files=no", "--", "src/flow_stp")
    assert importlib.metadata.version("flow-stp") == "0.0.0"
    return {"repo": str(repo), "import_file": str(Path(flow_stp.__file__).resolve()),
            "commit": PINNED_COMMIT, "method_version": METHOD_VERSION}


def _load_public():
    manifest = json.loads((DATA / "manifests/dataset_manifest.json").read_text())
    assert manifest["dataset_id"] == "synthetic_developmental_crescent_v1"
    assert manifest["seed"] == 20261023 and manifest["status"] == "frozen"
    assert manifest["n_observed_cells"] == 36425
    assert manifest["n_source_probes"] == 363
    assert manifest["genes"] == GENES
    assert np.isclose(float(manifest["source_t"]), 2/9)
    assert np.isclose(float(manifest["target_t"]), 2.5/9)
    public_paths = [
        "train_only/public/observed_cells.csv.gz",
        "train_only/public/counts.csv.gz",
        "train_only/public/genes.csv",
        "manifests/source_probe_cell_ids.csv.gz",
    ]
    hashes = {}
    for relative in public_paths:
        file = DATA / relative
        assert file.is_file()
        hashes[relative] = _sha(file)
        assert hashes[relative] == manifest["file_sha256"][relative], relative
    obs = pd.read_csv(DATA / public_paths[0])
    cdf = pd.read_csv(DATA / public_paths[1])
    genes = pd.read_csv(DATA / public_paths[2])
    probes = pd.read_csv(DATA / public_paths[3])
    assert list(obs.columns) == ["cell_id", "tissue_id", "time", "x", "y"]
    assert list(cdf.columns) == ["cell_id", *GENES]
    assert genes["gene"].tolist() == GENES
    assert len(obs) == len(cdf) == 36425
    obs["cell_id"] = obs["cell_id"].astype(str)
    assert obs["cell_id"].is_unique and cdf["cell_id"].is_unique
    np.testing.assert_array_equal(obs.cell_id.to_numpy(), cdf.cell_id.astype(str).to_numpy())
    np.testing.assert_array_equal(np.sort(obs.tissue_id.unique()), np.arange(10))
    np.testing.assert_allclose(obs.time.to_numpy(float), obs.tissue_id.to_numpy(int)/9,
                               atol=1e-10, rtol=0)
    assert not np.isclose(obs.time.to_numpy(float), 2.5/9).any()
    assert len(probes) == 363 and probes.cell_id.nunique() == 363
    assert set(probes.cell_id.astype(str)).issubset(
        set(obs.loc[obs.tissue_id.eq(2), "cell_id"]))
    counts = cdf[GENES].to_numpy(dtype=np.float32, copy=True)
    xy = obs[["x", "y"]].to_numpy(dtype=np.float32, copy=True)
    assert np.isfinite(counts).all() and (counts >= 0).all()
    assert np.array_equal(counts, np.rint(counts))
    assert np.isfinite(xy).all()
    count_slices, xy_slices = [], []
    for i in range(10):
        select = obs.tissue_id.to_numpy(int) == i
        count_slices.append(counts[select])
        xy_slices.append(xy[select])
    assert list(map(len, count_slices)) == [1800,2093,2419,2784,3193,3653,4171,4755,5411,6146]
    panel = pd.DataFrame({"slice_id": np.arange(10),
                          "tissue_id": np.arange(10),
                          "time": np.arange(10)/9.0})
    return obs, count_slices, xy_slices, panel, hashes


def _fisher_fields(model, z, t):
    """Exact directional derivatives of the repo's Fisher-flat field.

    torch.func.jvp changes *how* the derivatives are computed, not the model,
    Fisher-projection equation, eps, or RK4 integration rule. The implementation
    is checked below against the repo's native autograd implementation.
    """
    ones = torch.ones_like(z)
    zeros = torch.zeros_like(z)
    psi, psi_z = torch.func.jvp(model.fisher_flat, (z, t), (ones, zeros))
    _, psi_t = torch.func.jvp(model.fisher_flat, (z, t), (zeros, ones))
    energy = psi_z.square().sum(dim=1)
    v = -(psi_z * psi_t).sum(dim=1) / (energy + EPS)
    q = psi_t + v[:, None] * psi_z
    q_speed = torch.linalg.vector_norm(q, dim=1)
    return v, q_speed, energy, q, 0.25 * psi.square()


def _rk4_all_cells(model, initial, t_start, device, n_steps=STEPS):
    """Native RK4 formula on each observed cell, with q-speed accumulation.

    Flows stay in learned anatomical z, not physical x/y. At t=1 the
    zero-duration path is the identity. No intermediate unseen-time forecast.
    """
    n = len(initial)
    zpaths = np.empty((n, n_steps+1), dtype=np.float32)
    rpaths = np.empty_like(zpaths)
    zpaths[:, 0] = initial
    rpaths[:, 0] = 0.
    tt = np.linspace(t_start, 1., n_steps+1, dtype=np.float64)
    if np.isclose(t_start, 1.):
        zpaths[:] = initial[:, None]
        rpaths[:] = 0.
        return tt, zpaths, rpaths
    dt = (1.0-float(t_start))/n_steps
    for begin in range(0, n, CELL_BATCH):
        end = min(begin + CELL_BATCH, n)
        with torch.no_grad():
            z = torch.tensor(initial[begin:end], dtype=torch.float32, device=device)
            rho = torch.zeros_like(z)
            for step in range(n_steps):
                t0 = float(t_start) + step*dt
                t1 = torch.full_like(z, t0)
                k1z, k1r, *_ = _fisher_fields(model, z, t1)
                k2z, k2r, *_ = _fisher_fields(
                    model, z + dt*.5*k1z, torch.full_like(z, t0+dt*.5))
                k3z, k3r, *_ = _fisher_fields(
                    model, z + dt*.5*k2z, torch.full_like(z, t0+dt*.5))
                k4z, k4r, *_ = _fisher_fields(
                    model, z + dt*k3z, torch.full_like(z, t0+dt))
                z = z + dt/6*(k1z+2*k2z+2*k3z+k4z)
                rho = rho + dt/6*(k1r+2*k2r+2*k3r+k4r)
                if not torch.isfinite(z).all() or not torch.isfinite(rho).all():
                    raise FloatingPointError(f"Nonfinite Stage-2 path: tissue t={t_start}, step={step}")
                zpaths[begin:end, step+1] = z.cpu().numpy()
                rpaths[begin:end, step+1] = rho.cpu().numpy()
    return tt, zpaths, rpaths


def _crosscheck_native_stage2(model, device):
    # Compare forward-mode JVP derivatives against native reverse-mode autograd.
    # A small float32 evaluation tolerance accounts for different AD kernels.
    rng = np.random.default_rng(20261023)
    z = torch.tensor(rng.uniform(.15,.85,8), dtype=torch.float32, device=device)
    t = torch.tensor(rng.uniform(.05,.95,8), dtype=torch.float32, device=device)
    with torch.no_grad():
        v, q, energy, _, E = _fisher_fields(model, z, t)
    with torch.enable_grad():
        native = estimate_local_dynamics(model.fisher_flat, z, t, eps=EPS)
    np.testing.assert_allclose(v.cpu().numpy(), native.velocity.detach().reshape(-1).cpu().numpy(),
                               rtol=2e-3, atol=5e-4)
    native_q = native.residual_speed.detach().reshape(-1).cpu().numpy()
    quick_q = q.cpu().numpy()
    np.testing.assert_allclose(quick_q, native_q, rtol=5e-3, atol=2e-4)
    np.testing.assert_allclose(energy.cpu().numpy(), native.tangent_energy.detach().reshape(-1).cpu().numpy(),
                               rtol=1e-3, atol=2e-4)
    np.testing.assert_allclose(E.cpu().numpy(), native.expression.detach().cpu().numpy(),
                               rtol=1e-4, atol=2e-5)
    # Confirm RK4 path is the same as upstream native RK4 on a small public-only toy batch.
    z0 = torch.tensor([.2, .4, .65], dtype=torch.float32, device=device)
    with torch.enable_grad():
        native_path = integrate_flow_rk4(model.fisher_flat, z0,
                                         t0=2/9, t1=3/9, n_steps=8, eps=EPS)
    # Intra-interval independent fast RK4 replay (t=2/9 to t=3/9).
    zfast = z0.clone()
    dt = (1/9)/8
    with torch.no_grad():
        for step in range(8):
            ti = 2/9+step*dt
            a, *_ = _fisher_fields(model, zfast, torch.full_like(zfast, ti))
            b, *_ = _fisher_fields(model, zfast+.5*dt*a, torch.full_like(zfast,ti+.5*dt))
            c, *_ = _fisher_fields(model, zfast+.5*dt*b, torch.full_like(zfast,ti+.5*dt))
            d, *_ = _fisher_fields(model, zfast+dt*c, torch.full_like(zfast,ti+dt))
            zfast = zfast + dt/6*(a+2*b+2*c+d)
    native_end = native_path.final_position.detach().cpu().numpy()
    quick_end = zfast.cpu().numpy()
    np.testing.assert_allclose(quick_end, native_end, rtol=3e-3, atol=5e-4)
    return {
        "velocity_abs_error_max": float(np.max(np.abs(
            v.cpu().numpy() - native.velocity.detach().reshape(-1).cpu().numpy()))),
        "q_norm_abs_error_max": float(np.max(np.abs(quick_q - native_q))),
        "q_norm_relative_error_max": float(np.max(
            np.abs(quick_q - native_q) / np.maximum(np.abs(native_q), 1e-6))),
        "tangent_energy_abs_error_max": float(np.max(np.abs(
            energy.cpu().numpy() - native.tangent_energy.detach().reshape(-1).cpu().numpy()))),
        "rk4_endpoint_abs_error_max": float(np.max(np.abs(quick_end - native_end))),
        "q_norm_crosscheck_rtol": 5e-3,
        "q_norm_crosscheck_atol": 2e-4,
        "tf32_matmul_disabled_for_stage2": bool(
            torch.cuda.is_available() and not torch.backends.cuda.matmul.allow_tf32),
    }


def _run():
    source = _check_source()
    obs, counts, coords, panel, hashes = _load_public()
    cfg = Stage1ARunConfig(output_dir=NATIVE)  # every other field is exact repo default
    assert cfg.epochs == 1500 and cfg.batch_size_per_slice == 1024
    assert cfg.seed == 20260622 and cfg.use_library_size_offset
    assert cfg.coordinate_decoder == "shared_auto" and cfg.gauge_mode == "global"
    settings = {"dataset_id": "synthetic_developmental_crescent_v1",
                "source": source, "dataset_public_sha256": hashes,
                "stage1_run_config": asdict(cfg),
                "stage2": {"eps": EPS, "rk4_steps": STEPS,
                           "trajectory_end_time": 1.,
                           "rho_semantics": "integral of ||q|| from each cell's observed time to t, along native-z Fisher flow",
                           "derivatives": "torch.func.jvp, checked against flow_stp.stage2.estimate_local_dynamics"},
                "uses_hidden_truth": False, "heldout_forecast_generated": False}
    manifest_path = OUT / "run_config.json"
    done_path = OUT / "complete.json"
    if manifest_path.exists():
        saved = json.loads(manifest_path.read_text())
        assert saved == json.loads(json.dumps(settings, default=str)), (
            "Existing directory was fitted using different data/settings; do not overwrite.")
    elif OUT.exists() and any(OUT.iterdir()):
        raise RuntimeError(f"Unrecognized nonempty output directory: {OUT}")
    if done_path.exists():
        done = json.loads(done_path.read_text())
        assert done["source_commit"] == PINNED_COMMIT
        assert done["public_sha256"] == hashes
        for relative, sha in done["file_sha256"].items():
            p = OUT / relative
            assert p.is_file() and _sha(p) == sha, relative
        print(f"ALREADY COMPLETE | 10 tissues | 36,425 cells | {OUT}")
        return
    OUT.mkdir(parents=True, exist_ok=True)
    if not manifest_path.exists():
        _json(manifest_path, settings)
    log_path = OUT / "execution.log"
    console = sys.stdout
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        class TrainingTee(io.TextIOBase):
            def write(self, message):
                log.write(message)
                console.write(message)
                return len(message)

            def flush(self):
                log.flush()
                console.flush()

        with contextlib.redirect_stdout(TrainingTee()), contextlib.redirect_stderr(log):
            obs_for_runner = obs.copy()
            obs_for_runner["slice_id"] = obs_for_runner.tissue_id.astype(int)
            result = run_stage1a(counts_by_slice=counts, spatial_by_slice=coords,
                                 panel_df=panel, gene_names=GENES,
                                 obs_df=obs_for_runner, config=cfg)
            assert result.model.config.coordinate_decoder == "shared_time"
            assert (NATIVE / "done.json").is_file()
            assert json.loads((NATIVE / "done.json").read_text()).get("completed") is True
            assert len(result.cells) == len(obs) == 36425
            np.testing.assert_allclose(result.normalized_times,
                                       np.arange(10)/9, atol=2e-7, rtol=0)
            model = result.model.eval()
            device = next(model.parameters()).device
            # Training uses the upstream CUDA configuration. Disable TF32 only
            # for readout: this improves agreement between forward- and
            # reverse-mode derivatives without changing fitted parameters.
            if device.type == "cuda":
                torch.backends.cuda.matmul.allow_tf32 = False
                torch.backends.cudnn.allow_tf32 = False
            crosscheck = _crosscheck_native_stage2(model, device)
            _json(OUT / "stage2_derivative_crosscheck.json", crosscheck)
            row = obs.copy()
            row["slice_id"] = row.tissue_id.astype(int)
            row["cell_index"] = row.groupby("slice_id", sort=False).cumcount()
            row = row.merge(result.cells[["slice_id", "cell_index", "z_stage1a"]],
                            on=["slice_id", "cell_index"], how="left",
                            sort=False, validate="one_to_one")
            np.testing.assert_array_equal(row.cell_id.to_numpy(), obs.cell_id.to_numpy())
            assert row.z_stage1a.notna().all()
            z = row.z_stage1a.to_numpy(dtype=np.float32)
            assert np.isfinite(z).all()
            v_all = np.empty(len(row), np.float32)
            q_all = np.empty(len(row), np.float32)
            tangent_all = np.empty(len(row), np.float32)
            E_all = np.empty((len(row),48), np.float32)
            qvec_all = np.empty_like(E_all)
            for a in range(0, len(row), CELL_BATCH):
                b = min(a+CELL_BATCH, len(row))
                zz = torch.tensor(z[a:b], device=device, dtype=torch.float32)
                tt = torch.tensor(row.time.to_numpy(dtype=np.float32)[a:b], device=device)
                with torch.no_grad():
                    v, q, tang, qvec, E = _fisher_fields(model, zz, tt)
                v_all[a:b] = v.cpu().numpy()
                q_all[a:b] = q.cpu().numpy()
                tangent_all[a:b] = tang.cpu().numpy()
                E_all[a:b] = E.cpu().numpy()
                qvec_all[a:b] = qvec.cpu().numpy()
            assert all(np.isfinite(x).all() for x in (v_all,q_all,tangent_all,E_all,qvec_all))
            assert (tangent_all >= 0).all() and (E_all > 0).all()
            row["V_learned_z"] = v_all
            row["q_norm"] = q_all
            row["fisher_tangent_energy"] = tangent_all
            row["E_total"] = E_all.sum(axis=1)
            row.to_csv(OUT / "stage2_all_observed_cells.csv.gz", index=False,
                       compression="gzip")
            np.savez_compressed(OUT / "stage2_observed_cell_fields.npz",
                                cell_id=row.cell_id.to_numpy(str),
                                gene_names=np.asarray(GENES, dtype=str),
                                tissue_id=row.tissue_id.to_numpy(np.int32),
                                time=row.time.to_numpy(np.float32),
                                z_learned=z, V=v_all, q_norm=q_all,
                                q_components=qvec_all, tangent_energy=tangent_all,
                                E=E_all)
            # Integrate learned-z trajectories from observed times to t=1.
            flow_dir = OUT / "stage2_native_z_flow_by_tissue"
            flow_dir.mkdir(exist_ok=True)
            for s in range(10):
                mask = row.tissue_id.to_numpy(int) == s
                z_s = z[mask]
                ts, path, progress = _rk4_all_cells(model, z_s, s/9, device)
                assert path.shape == progress.shape == (mask.sum(), STEPS+1)
                assert np.all(progress >= -1e-4)  # integral of nonnegative q norm
                np.savez_compressed(flow_dir / f"tissue_{s:02d}.npz",
                                    cell_id=row.cell_id.to_numpy(str)[mask],
                                    start_time=np.array(s/9),
                                    time=ts.astype(np.float32),
                                    source_learned_z=z_s,
                                    F_learned_z=path,
                                    rho_from_snapshot=progress)
            hashes_out = {}
            for f in sorted(OUT.rglob("*")):
                if f.is_file() and f.name not in {"complete.json", "execution.log"}:
                    hashes_out[f.relative_to(OUT).as_posix()] = _sha(f)
            _json(done_path, {"completed": True,
                              "source_commit": PINNED_COMMIT,
                              "method_version": METHOD_VERSION,
                              "public_sha256": hashes,
                              "n_tissues": 10, "n_cells": 36425,
                              "n_genes": 48, "stage2_time_range": [0.,1.],
                              "no_unseen_time_expression_forecast": True,
                              "stage2_derivative_crosscheck": "passed",
                              "file_sha256": hashes_out})
    print(f"COMPLETE | Stage 1 + Stage 2 | 10 tissues | 36,425 cells | {OUT}")


_run()